# 26장 만든 모델 보여주기 — Gradio

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `app.py` 와 `test_app.py` 를 치고, `test_app.py` 먼저 → `app.py` 실행 → 브라우저에서 그리기 | [ch26_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_1_follow.ipynb) |
| 2. 주석 달기 | `predict` 각 줄에 텐서 shape과 "왜 반전하는가"를 주석으로 | [ch26_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `predict` 복원. 그다음 확인 문제 5(조문 분류기 화면) | [ch26_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "26"))                      # 이 장의 코드 폴더로 이동
if "google.colab" in sys.modules:
    !pip install -q gradio
if not os.path.exists("../13/results/lr01/model_best.pt"):      # 13장에서 학습한 모델이 없으면 만든다
    !cd ../13 && python train.py --run lr01
print("준비 끝. 작업 폴더: code/26")

## `app.py` — 대상: `predict`

In [ ]:
%%writefile app.py
# 26장 만든 모델 보여주기: 13장에서 학습한 MNIST 분류기를 그림판이 달린 웹 화면으로
# 준비: pip install gradio      실행: python app.py  → 브라우저에서 http://127.0.0.1:7860
import os
import sys
import json
import numpy as np
import torch
import gradio as gr
from PIL import Image

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "13"))
from train import MLP                                     # 13장의 모델 정의를 그대로

RUN_DIR = os.path.join(os.path.dirname(__file__), "..", "13", "results", "lr01")


def load_model():
    with open(os.path.join(RUN_DIR, "config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    m = MLP(hidden=cfg["hidden"])
    m.load_state_dict(torch.load(os.path.join(RUN_DIR, "model_best.pt"), map_location="cpu"))
    m.eval()
    return m


model = load_model()


def predict(image):
    """그림판 입력(딕셔너리 또는 배열) → 28x28 흑백 → 확률 10개"""
    # Sketchpad 는 {"composite": 배열} 형태
    # 흰 바탕에 검은 글씨면 반전 (MNIST 는 검은 바탕)
    raise NotImplementedError("여기를 채우세요")


demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(label="숫자를 그려 보세요", image_mode="L"),
    outputs=gr.Label(num_top_classes=3, label="예측"),
    title="손글씨 숫자 분류기 (12·13장 MLP)",
    description="13장에서 학습한 모델(시험 정확도 94.7%)입니다. 그림판에 0~9를 그리면 확률을 보여 줍니다.",
)

if __name__ == "__main__":
    demo.launch()                                          # share=True 를 주면 1주일짜리 공개 링크가 생긴다

### 확인

In [ ]:
!python test_app.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- app.py

## 그다음

확인 문제 5(조문 분류기 화면)를 만들어 보세요.

In [ ]:
# 여기에 이어서 작성하세요